Token classification encompasses any problem that can be formulated as “attributing a label to each token in a sentence,” such as:

- **Named entity recognition (NER)**: Find the entities (such as persons, locations, or organizations) in a sentence. This can be formulated as attributing a label to each token by having one class per entity and one class for “no entity.”
- **Part-of-speech tagging (POS)**: Mark each word in a sentence as corresponding to a particular part of speech (such as noun, verb, adjective, etc.).
- **Chunking**: Find the tokens that belong to the same entity. This task (which can be combined with POS or NER) can be formulated as attributing one label (usually `B-`) to any tokens that are at the beginning of a chunk, another label (usually `I-`) to tokens that are inside a chunk, and a third label (usually `O`) to tokens that don’t belong to any chunk.

# Load dependencies

In [ ]:
from datasets import load_dataset, load_dataset_builder
from transformers import AutoTokenizer

# Config

In [ ]:
user_name = "amin-oj"

dataset_dict = {
    "path": "conll2003",
    "revision": "refs/convert/parquet"
}
model_checkpoint = "bert-base-cased"
push_to_hub=True
train_bs = 16
eval_bs = 16
num_train_epochs = 3
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-ner-{dataset_dict['path']}"
print(ckpt_name)

# HF Login

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!hf auth whoami

# Load data

In [ ]:
# dsb = load_dataset_builder(**dataset_dict)
# for k,v in dsb.__dict__.items():
#   print(f"=================== {k} ===================")
#   print(f"{v}")

In [ ]:
raw_datasets = load_dataset(**dataset_dict)

# Look at data

In [ ]:
print(raw_datasets)
print("====================")

for k,v in raw_datasets["train"].features.items():
  print(f"feature name: {k}")
  print(f"feature type: {v}")
print("====================")

In [ ]:
first_sample = raw_datasets["train"][3]
print(first_sample["tokens"])
print("====================")
print(first_sample["ner_tags"])
print("====================")
print(first_sample["pos_tags"])
print("====================")
print(first_sample["chunk_tags"])
print("====================")


ner_feature = raw_datasets["train"].features["ner_tags"]
label_names = ner_feature.feature.names
words = first_sample["tokens"]
labels = first_sample["ner_tags"]
line1 = ""
line2 = ""
for word, label in zip(words, labels):
    full_label = label_names[label]
    max_length = max(len(word), len(full_label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += full_label + " " * (max_length - len(full_label) + 1)

print(line1)
print(line2)

- `"tokens"` are pre-tokenized
- contains labels for all three **POS, NER, and chunking** tasks
- NER tag meaning:
  -   `O` means the word doesn’t correspond to any entity.
  -   `B-PER`/`I-PER` means the word corresponds to the beginning of/is inside a _person_ entity.
  -   `B-ORG`/`I-ORG` means the word corresponds to the beginning of/is inside an _organization_ entity.
  -   `B-LOC`/`I-LOC` means the word corresponds to the beginning of/is inside a _location_ entity.
  -   `B-MISC`/`I-MISC` means the word corresponds to the beginning of/is inside a _miscellaneous_ entity.

# Preprocess data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
assert tokenizer.is_fast

In [ ]:
inputs = tokenizer(first_sample["tokens"], is_split_into_words=True)
print(inputs.tokens())
print("==============")
print(inputs.word_ids())
print("==============")
print(first_sample['ner_tags'])
print("==============")

- We need to match the tokens with the labels
  - we have one lable per word
  - but possibly more than one token per word

- The first rule we’ll apply is that special tokens get a label of `-100`. This is because by default `-100` is an index that is ignored in the loss function we will use (cross entropy).
- Then, each token gets the same label as the token that started the word it’s inside, since they are part of the same entity.
  - Some researchers prefer to attribute only one label per word, and assign -100 to the other subtokens in a given word.
    - This is to avoid long words that split into lots of subtokens contributing heavily to the loss.
- For tokens inside a word but not at the beginning, we replace the `B-` with `I-` (since the token does not begin the entity)

In [ ]:
def align_labels_with_tokens(labels, word_ids, first_token_only = False):
  new_labels = []
  current_word = None
  for word_id in word_ids:
    if word_id != current_word:
      # Start of a new word!
      current_word = word_id
      label = -100 if word_id is None else labels[word_id]
      new_labels.append(label)
    elif word_id is None:
      # Special token
      new_labels.append(-100)
    else:
      if first_token_only:
        label = -100
      else:
        # Same word as previous token
        label = labels[word_id]
        # If the label is B-XXX we change it to I-XXX
        if label % 2 == 1:
          label += 1
      new_labels.append(label)

  return new_labels

word_ids = inputs.word_ids()
print(labels)
print("====================")
print(align_labels_with_tokens(labels, word_ids))
print("====================")
print(align_labels_with_tokens(labels, word_ids, first_token_only=True))
print("====================")

## Define the batched prep function

In [ ]:
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=False, # the default
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    fn_kwargs={"tokenizer": tokenizer},
    desc="Running tokenizer on dataset",
)

print(tokenized_datasets)

# Fine-tuning the model with the Trainer API

## Data collation

- We can’t just use a `DataCollatorWithPadding` because that only pads the inputs (input IDs, attention mask, and token type IDs).
- Here our labels should be padded the exact same way as the inputs so that they stay the same size, using `-100` as a value so that the corresponding predictions are ignored in the loss computation.
- This is all done by a [`DataCollatorForTokenClassification`](https://huggingface.co/transformers/main_classes/data_collator.html#datacollatorfortokenclassification).

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print(tokenized_datasets["train"][0])
print("===================")
print(tokenized_datasets["train"][1])
print("===================")
batch = data_collator([tokenized_datasets["train"][i] for i in range(2)])
print(batch)
print("===================")
# TODO: why didn't we collate labels in hf_trainer_basic.ipynb???

## Metrics

In [ ]:
!pip install -q seqeval evaluate
import evaluate
metric = evaluate.load("seqeval")

- This metric does not behave like the standard accuracy: it will actually take the lists of labels as strings, not integers.
- So, we will need to fully decode the predictions and labels before passing them to the metric.
- Note that the metric takes a list of predictions (not just one) and a list of labels.

In [ ]:
labels = raw_datasets["train"][0]["ner_tags"]
labels = [label_names[i] for i in labels]
print(labels)
print("================")
predictions = labels.copy()
predictions[2] = "O"
result = metric.compute(predictions=[predictions], references=[labels])
for key, value in result.items():
  print(f"{key}: {value}")
print("================")

In [ ]:
import numpy as np
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

## Defining the model

- Creating the model issues a warning that
  - some weights were not used (the ones from the pretraining head)
  - and some other weights are randomly initialized (the ones from the new token classification head), and that this model should be trained.
- ⚠️ If you have a model with the wrong number of labels, you will get an obscure error when calling the `Trainer.train()` method later on (something like “CUDA error: device-side assert triggered”). This is the number one cause of bugs reported by users for such errors, so make sure you do this check to confirm that you have the expected number of labels.

In [ ]:
from transformers import AutoModelForTokenClassification

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

print("==============")
print(f"number of labels: {model.config.num_labels}")
print("==============")

## Fine-tuning the model

- Note that you can specify the name of the repository you want to push to with the `hub_model_id` argument (in particular, you will have to use this argument to push to an organization).
- Note that while the training happens, each time the model is saved (here, every epoch) it is uploaded to the Hub in the background. This way, you will be able to to resume your training on another machine if necessary.

In [ ]:
from transformers import TrainingArguments
from transformers import Trainer

args = TrainingArguments(
  output_dir=ckpt_name,
  push_to_hub=push_to_hub,
  per_device_train_batch_size=train_bs,
  per_device_eval_batch_size=eval_bs,
  num_train_epochs=num_train_epochs,
  learning_rate=2e-5,
  weight_decay=0.01,
  eval_strategy="epoch",
  save_strategy="epoch",
  load_best_model_at_end=True,
  metric_for_best_model="f1",
  greater_is_better=True,
  report_to="none" # to disable wandb login
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
trainer.push_to_hub(commit_message="Training complete")

The `Trainer` also drafts a model card with all the evaluation results and uploads it.